In [32]:
from langchain_core.tools import tool
from openai import OpenAI
from ddgs import DDGS
from langgraph.graph import *
from dotenv import load_dotenv
from typing import TypedDict, Annotated

import os


load_dotenv()

API_KEY = os.getenv('API_KEY')
API_URL = "https://api.groq.com/openai/v1"
open_ai_client = OpenAI(api_key=API_KEY, base_url=API_URL)

SUPPORT METHOD

In [33]:
def do_api_call(system_role:str, system_message:str, user_role:str, user_message:str) ->str:
    completion= open_ai_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{
            "role": system_role,
            "content": system_message,
        },
            {
                "role": user_role,
                "content": user_message,
            }
        ],
        temperature=0.1,
    )
    return completion.choices[0].message.content


**TOOLS**

In [34]:
def do_google_search(
    query: str,
    max_results: int = 10
):
    try:
        results = []

        with DDGS() as ddgs:
            search_results = ddgs.text(
                query=query,
                backend="google",   # Use Google
                max_results=max_results
            )

            for item in search_results:
                results.append({
                    "title": item.get("title"),
                    "url": item.get("href"),
                    "description": item.get("body")
                })

        return results

    except Exception as e:
        print(f"Error: {e}")
        return []

In [35]:
# data = google_search(
#     "today news about gold",
#     max_results=5
# )
#
# for i, item in enumerate(data, start=1):
#     print(f"\nResult {i}")
#     print("Title:", item["title"])
#     print("URL:", item["url"])
#     print("Description:", item["description"])
#     print("-" * 50)

In [36]:
class AgentState(TypedDict):
    input: str
    status: str
    stage: int
    response: dict
    stage_name: str

In [37]:
SUMMARY_PROMPT = """
You are a summary agent.

Your responsibility is to give a summary
of the search results.

Search Results:
{response}
"""

In [38]:
def do_start_processing(
    input_query: AgentState
) -> AgentState:

    # Google search
    search_response = do_google_search(
        input_query["input"],
        5
    )
    print(search_response)

    formatted_results = "\n\n".join([
    f"Title: {item['title']}\n"
    f"Description: {item['description']}"
    for item in search_response
])

    # Format prompt
    user_msg = SUMMARY_PROMPT.format(
    response=formatted_results
)

    # System prompt
    sys_msg = """
    Answer all user queries.

    If user asks about any model-related query,
    say a sarcastic story.
    """

    # LLM API call
    response = do_api_call(
        system_role="system",
        system_message=sys_msg,
        user_role="user",
        user_message=user_msg
    )

    return {
        "status": "Completed",
        "stage": 1,
        "response": response,
        "stage_name": "Summary response"
    }

In [39]:
from typing import TypedDict
from langgraph.graph import (
    StateGraph,
    START,
    END
)
# Graph
init_graph = StateGraph(AgentState)

init_graph.add_node(
    "answering_node",
    do_start_processing
)

init_graph.add_edge(
    START,
    "answering_node"
)

init_graph.add_edge(
    "answering_node",
    END
)

graph = init_graph.compile()

# Input
input_data = {
    "input": "Search about today's trending news"
}

# Invoke
result = graph.invoke(input_data)

print(result)

[{'title': 'Earlier today in the Gulf of Oman, U.S. Marines from the 31st Marine ...', 'url': 'https://x.com/CENTCOM/status/2057144264526598381', 'description': '3 days ago · ... have now redirected 91 commercial ships to ensure compliance. 285. 2268. 13198. 833232 · · Explore Trending StoriesGo to HomeSearch XNews.'}, {'title': 'Celebrity Makeup Artist Shares 3 Trending Looks for Summer 2026', 'url': 'https://www.today.com/video/celebrity-makeup-artist-shares-3-trending-looks-for-summer-2026-263516229680', 'description': "5 days ago · ... TODAY's Jenna Bush Hager and guest co-host Carson Daly to show three ... search by queryly. Advanced Search close. Company Logo. Residents ..."}, {'title': "See TODAY's Savannah Guthrie and Al Roker's Cameos in 'Hacks'", 'url': 'https://www.today.com/video/see-today-s-savannah-guthrie-and-al-roker-s-cameos-in-hacks-263788101978', 'description': '1 day ago · As the Emmy Award-winning HBO Max series "Hacks" wraps up its 5-year run, TODAY\'s own Savanna